In [ ]:
# ============================================================
# South Africa AML Context Evidence Plots
# Cash intensity, informal economy, socioeconomic pressures,
# and AML false-positive context
# Publication-style dashboard version
# ============================================================

import sys
from pathlib import Path

# Keep imports working when this notebook is run from notebooks/.
PROJECT_ROOT_CANDIDATE = Path.cwd()
if PROJECT_ROOT_CANDIDATE.name.lower() == "notebooks":
    PROJECT_ROOT_CANDIDATE = PROJECT_ROOT_CANDIDATE.parent
if str(PROJECT_ROOT_CANDIDATE) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_CANDIDATE))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import textwrap

from src.sa_aml_simulation.paths import AML_CONTEXT_PLOTS_DIR, ensure_project_dirs

# -----------------------------
# Output folder
# -----------------------------
ensure_project_dirs()
OUT_DIR = AML_CONTEXT_PLOTS_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving socioeconomic context plots to: {OUT_DIR}")

# -----------------------------
# Plot style settings
# -----------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "figure.dpi": 120,
    "savefig.dpi": 300
})

# -----------------------------
# Colour palette
# -----------------------------
NAVY = "#132A5E"
BLUE = "#2563EB"
CYAN = "#06B6D4"
TEAL = "#0D9488"
GREEN = "#16A34A"
ORANGE = "#F97316"
AMBER = "#F59E0B"
RED = "#DC2626"
PINK = "#DB2777"
PURPLE = "#7C3AED"
SLATE = "#475569"
GREY = "#64748B"
LIGHT_GREY = "#F1F5F9"
DARK = "#111827"

cash_colors = [BLUE, CYAN, AMBER]
informal_colors = [TEAL, GREEN, AMBER, ORANGE]
consumer_colors = [BLUE, CYAN, PURPLE, PINK]

# -----------------------------
# Helper functions
# -----------------------------
def clean_axis(ax, grid_axis="y"):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CBD5E1")
    ax.spines["bottom"].set_color("#CBD5E1")
    ax.tick_params(axis="both", colors="#334155")
    ax.grid(axis=grid_axis, alpha=0.22, linestyle="-", linewidth=0.7)
    ax.set_axisbelow(True)

def add_source_note(fig, text):
    fig.text(
        0.01, 0.01, text,
        ha="left", va="bottom",
        fontsize=8.3, color=GREY
    )

def add_bar_labels_vertical(ax, bars, labels, offset=1.0, fontsize=9.5):
    for bar, label in zip(bars, labels):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            label,
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color=DARK,
            fontweight="bold"
        )

def add_bar_labels_horizontal(ax, bars, labels, offset=1.0, fontsize=9.5):
    for bar, label in zip(bars, labels):
        ax.text(
            bar.get_width() + offset,
            bar.get_y() + bar.get_height() / 2,
            label,
            ha="left",
            va="center",
            fontsize=fontsize,
            color=DARK,
            fontweight="bold"
        )

def make_card(ax, title, value, subtitle, color):
    """
    Dashboard KPI card.
    This version prevents long titles from overlapping with the KPI value.
    """
    ax.set_facecolor(LIGHT_GREY)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_xticks([])
    ax.set_yticks([])

    wrapped_title = textwrap.fill(title, width=24)
    wrapped_subtitle = textwrap.fill(subtitle, width=30)

    title_lines = wrapped_title.count("\n") + 1

    if title_lines >= 2:
        title_fontsize = 9.0
        value_y = 0.38
        value_fontsize = 21
        subtitle_y = 0.10
    else:
        title_fontsize = 10.5
        value_y = 0.47
        value_fontsize = 24
        subtitle_y = 0.16

    ax.text(
        0.06, 0.82, wrapped_title,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=title_fontsize,
        color=SLATE,
        fontweight="bold",
        linespacing=0.95
    )

    ax.text(
        0.06, value_y, value,
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=value_fontsize,
        color=color,
        fontweight="bold"
    )

    ax.text(
        0.06, subtitle_y, wrapped_subtitle,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.4,
        color=GREY,
        linespacing=0.95
    )

# ============================================================
# Data
# ============================================================

cash_df = pd.DataFrame({
    "holder": [
        "Individuals and non-bank\nfinancial players",
        "South African\nReserve Bank",
        "Commercial\nbanks"
    ],
    "amount_rbn": [118, 34, 28]
})
cash_df["share_percent"] = cash_df["amount_rbn"] / cash_df["amount_rbn"].sum() * 100

grants_df = pd.DataFrame({
    "indicator": [
        "Individuals receiving\nsocial grants",
        "Households receiving\nsocial grants"
    ],
    "percent": [28.2, 39.1],
    "group": ["Social grants", "Social grants"]
})

unemployment_df = pd.DataFrame({
    "indicator": [
        "Official\nunemployment",
        "Expanded / combined\nunemployment"
    ],
    "percent": [31.4, 42.1],
    "group": ["Labour market", "Labour market"]
})

informal_df = pd.DataFrame({
    "indicator": [
        "Informal economy\nshare of GNP",
        "Informal economy\nshare of employment",
        "Informal businesses\nunregistered",
        "Access to\nbank loans"
    ],
    "percent": [12.3, 19.5, 80.0, 9.0],
    "label": ["12.3%", "19.5%", "~80%", "<9%"]
})

cash_behaviour_df = pd.DataFrame({
    "indicator": [
        "Used cash for\nfood/groceries",
        "Use cash most often\nfor food/groceries",
        "Prefer cash\nin future",
        "Use both cash\nand digital"
    ],
    "percent": [99.0, 71.0, 64.0, 96.0],
    "label": ["99%", "71%", "64%", "96%"]
})

context_df = pd.DataFrame({
    "factor": [
        "Cash outside\ncommercial banks",
        "Informal businesses\nunregistered",
        "Used cash for\nfood/groceries",
        "Prefer cash\nin future",
        "Households receiving\ngovernment grants",
        "Expanded / combined\nunemployment"
    ],
    "percent": [
        cash_df.loc[
            cash_df["holder"] == "Individuals and non-bank\nfinancial players",
            "share_percent"
        ].iloc[0],
        80.0,
        99.0,
        64.0,
        39.1,
        42.1
    ],
    "label": ["65.6%", "~80%", "99%", "64%", "39.1%", "42.1%"]
}).sort_values("percent", ascending=True)

# ============================================================
# Plot 1: Cash distribution donut chart
# ============================================================
fig, ax = plt.subplots(figsize=(8, 8), facecolor="white")

wedges, texts, autotexts = ax.pie(
    cash_df["amount_rbn"],
    labels=cash_df["holder"],
    autopct=lambda pct: f"{pct:.1f}%\nR{pct/100*cash_df['amount_rbn'].sum():.0f}bn",
    startangle=90,
    pctdistance=0.74,
    colors=cash_colors,
    wedgeprops=dict(width=0.50, edgecolor="white", linewidth=3),
    textprops=dict(color=DARK, fontsize=10)
)

for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight("bold")
    autotext.set_color("white")

ax.text(
    0, 0,
    "R180bn\nTotal cash\nin circulation",
    ha="center",
    va="center",
    fontsize=13,
    color=NAVY,
    fontweight="bold"
)

ax.set_title(
    "Where is South Africa's cash?\nNotes and coins in circulation, 2024",
    fontsize=15,
    color=DARK,
    pad=18,
    fontweight="bold"
)

add_source_note(
    fig,
    "Source: PayInc, reported by BusinessTech / The Outlier. Values shown in Rand billion."
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(OUT_DIR / "sa_cash_distribution_donut.png", bbox_inches="tight")
plt.show()

# ============================================================
# Plot 2: Cash distribution bar chart
# ============================================================
fig, ax = plt.subplots(figsize=(10.8, 6.4), facecolor="white")

bars = ax.bar(
    cash_df["holder"],
    cash_df["amount_rbn"],
    color=cash_colors,
    edgecolor="white",
    linewidth=1.5,
    width=0.62
)

ax.set_title(
    "Cash distribution by holder in South Africa, 2024",
    fontsize=15,
    color=DARK,
    fontweight="bold",
    pad=16
)

ax.text(
    -0.5,
    max(cash_df["amount_rbn"]) + 16,
    "Individuals and non-bank financial players hold the largest share of cash in circulation.",
    fontsize=10.5,
    color=GREY
)

ax.set_ylabel("Amount held (R billion)")
ax.set_xlabel("")
ax.set_ylim(0, max(cash_df["amount_rbn"]) + 28)

labels = [
    f"R{value}bn\n{share:.1f}%"
    for value, share in zip(cash_df["amount_rbn"], cash_df["share_percent"])
]
add_bar_labels_vertical(ax, bars, labels, offset=2.2)

clean_axis(ax)
ax.grid(axis="x", visible=False)

add_source_note(
    fig,
    "Source: PayInc, reported by BusinessTech / The Outlier. Total cash in circulation: R180bn."
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(OUT_DIR / "sa_cash_distribution_bar.png", bbox_inches="tight")
plt.show()

# ============================================================
# Plot 3: Social grants and unemployment indicators
# ============================================================
combined_df = pd.concat([
    grants_df.rename(columns={"indicator": "metric", "percent": "value"}),
    unemployment_df.rename(columns={"indicator": "metric", "percent": "value"})
], ignore_index=True)

fig, ax = plt.subplots(figsize=(11.5, 6.5), facecolor="white")

x = np.arange(len(combined_df))
bars = ax.bar(
    x,
    combined_df["value"],
    color=[BLUE, CYAN, ORANGE, RED],
    edgecolor="white",
    linewidth=1.5,
    width=0.58
)

ax.set_xticks(x)
ax.set_xticklabels(combined_df["metric"])
ax.set_ylabel("Share of population / labour force (%)")
ax.set_ylim(0, max(combined_df["value"]) + 13)

ax.set_title(
    "Socioeconomic indicators shaping South African transaction behaviour",
    fontsize=15,
    color=DARK,
    fontweight="bold",
    pad=16
)

ax.text(
    -0.45,
    max(combined_df["value"]) + 7,
    "Grant dependence and unemployment contribute to irregular income patterns and cash-based activity.",
    fontsize=10.5,
    color=GREY
)

labels = [f"{value:.1f}%" for value in combined_df["value"]]
add_bar_labels_vertical(ax, bars, labels, offset=1.2)

clean_axis(ax)
ax.grid(axis="x", visible=False)

for i, group in enumerate(combined_df["group"]):
    ax.text(
        i,
        -6.5,
        group,
        ha="center",
        va="top",
        fontsize=9,
        color=GREY
    )

add_source_note(
    fig,
    "Sources: Stats SA General Household Survey 2024; Stats SA Quarterly Labour Force Survey Q4 2025."
)

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(OUT_DIR / "sa_socioeconomic_indicators.png", bbox_inches="tight")
plt.show()

# ============================================================
# Plot 4: AML context factors horizontal bar chart
# ============================================================
fig, ax = plt.subplots(figsize=(12.2, 6.9), facecolor="white")

bars = ax.barh(
    context_df["factor"],
    context_df["percent"],
    color=[BLUE, CYAN, GREEN, ORANGE, RED, PURPLE],
    edgecolor="white",
    linewidth=1.5,
    height=0.6
)

ax.set_title(
    "Structural context factors relevant to AML false positives in South Africa",
    fontsize=15,
    color=DARK,
    fontweight="bold",
    pad=16
)

ax.text(
    0,
    len(context_df) - 0.25,
    "Cash, informality and socioeconomic pressure may generate legitimate behaviour that resembles AML risk signals.",
    fontsize=10.5,
    color=GREY
)

ax.set_xlabel("Percentage (%)")
ax.set_xlim(0, max(context_df["percent"]) + 14)

labels = context_df["label"].tolist()
add_bar_labels_horizontal(ax, bars, labels, offset=1.2)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#CBD5E1")
ax.tick_params(axis="y", length=0, colors="#334155")
ax.tick_params(axis="x", colors="#334155")
ax.grid(axis="x", alpha=0.22, linestyle="-", linewidth=0.7)
ax.set_axisbelow(True)

add_source_note(
    fig,
    "Sources: PayInc / BusinessTech; SARB Retail CBDC Background Note 2025; SBV Consumer Cash Survey 2024; Stats SA GHS 2024; Stats SA QLFS Q4 2025."
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUT_DIR / "sa_aml_context_factors.png", bbox_inches="tight")
plt.show()

# ============================================================
# Plot 5: Informal economy evidence
# ============================================================
fig, ax = plt.subplots(figsize=(11.8, 6.8), facecolor="white")

x = np.arange(len(informal_df))
bars = ax.bar(
    x,
    informal_df["percent"],
    color=informal_colors,
    edgecolor="white",
    linewidth=1.5,
    width=0.60
)

ax.set_xticks(x)
ax.set_xticklabels(informal_df["indicator"])
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 92)

ax.set_title(
    "Informal economy indicators relevant to AML context modelling",
    fontsize=15,
    color=DARK,
    fontweight="bold",
    pad=16
)

ax.text(
    -0.45,
    85,
    "Informal businesses often operate outside formal registration and formal credit channels.",
    fontsize=10.5,
    color=GREY
)

add_bar_labels_vertical(ax, bars, informal_df["label"].tolist(), offset=1.5)

clean_axis(ax)
ax.grid(axis="x", visible=False)

add_source_note(
    fig,
    "Source: BusinessTech reporting on Standard Bank Township Informal Economy Research."
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUT_DIR / "sa_informal_economy_indicators.png", bbox_inches="tight")
plt.show()

# ============================================================
# Plot 6: Cash behaviour evidence
# ============================================================
fig, ax = plt.subplots(figsize=(11.8, 6.8), facecolor="white")

x = np.arange(len(cash_behaviour_df))
bars = ax.bar(
    x,
    cash_behaviour_df["percent"],
    color=consumer_colors,
    edgecolor="white",
    linewidth=1.5,
    width=0.60
)

ax.set_xticks(x)
ax.set_xticklabels(cash_behaviour_df["indicator"])
ax.set_ylabel("Share of consumers (%)")
ax.set_ylim(0, 108)

ax.set_title(
    "Consumer cash behaviour supporting South Africa's cash-heavy AML context",
    fontsize=15,
    color=DARK,
    fontweight="bold",
    pad=16
)

ax.text(
    -0.45,
    101,
    "Cash remains important even where digital payments coexist with cash usage.",
    fontsize=10.5,
    color=GREY
)

add_bar_labels_vertical(ax, bars, cash_behaviour_df["label"].tolist(), offset=1.5)

clean_axis(ax)
ax.grid(axis="x", visible=False)

add_source_note(
    fig,
    "Sources: SARB Retail CBDC Background Note 2025; SBV Consumer Cash Survey White Paper 2024."
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUT_DIR / "sa_cash_behaviour_indicators.png", bbox_inches="tight")
plt.show()

# ============================================================
# Plot 7: Combined dashboard-style view
# ============================================================
fig = plt.figure(figsize=(17, 11), facecolor="white")
gs = fig.add_gridspec(
    3, 4,
    height_ratios=[1.05, 2.2, 2.2],  # slightly taller KPI cards
    width_ratios=[1.22, 1, 1, 1],
    hspace=0.55,
    wspace=0.48
)

# Cards
ax_card1 = fig.add_subplot(gs[0, 0])
ax_card2 = fig.add_subplot(gs[0, 1])
ax_card3 = fig.add_subplot(gs[0, 2])
ax_card4 = fig.add_subplot(gs[0, 3])

make_card(ax_card1, "Cash in circulation", "R180bn", "Notes and coins, 2024", BLUE)
make_card(ax_card2, "Informal economy", "R900bn", "Estimated annual value", TEAL)
make_card(ax_card3, "Cash for groceries", "99%", "Adults using cash", ORANGE)
make_card(ax_card4, "Unregistered informal businesses", "~80%", "Township informal economy survey", RED)

# Dashboard donut chart
ax1 = fig.add_subplot(gs[1:, 0])

wedges, texts, autotexts = ax1.pie(
    cash_df["amount_rbn"],
    labels=None,
    autopct=lambda pct: f"{pct:.1f}%",
    startangle=90,
    pctdistance=0.72,
    colors=cash_colors,
    wedgeprops=dict(width=0.52, edgecolor="white", linewidth=2.2),
    textprops=dict(color="white", fontsize=9.5, fontweight="bold")
)

for autotext in autotexts:
    autotext.set_fontsize(9.5)
    autotext.set_fontweight("bold")
    autotext.set_color("white")

ax1.text(
    0, 0,
    "R180bn\ncash",
    ha="center",
    va="center",
    fontsize=12,
    color=NAVY,
    fontweight="bold"
)

ax1.set_title(
    "Cash circulation split",
    fontsize=13,
    fontweight="bold",
    color=DARK,
    pad=14
)

ax1.text(
    -1.45, -0.42,
    "Individuals and\nnon-bank financial\nplayers\nR118bn",
    ha="right", va="center",
    fontsize=8.8, color=DARK
)

ax1.text(
    1.26, -0.02,
    "South African\nReserve Bank\nR34bn",
    ha="left", va="center",
    fontsize=8.8, color=DARK
)

ax1.text(
    0.42, 1.00,
    "Commercial\nbanks\nR28bn",
    ha="left", va="bottom",
    fontsize=8.8, color=DARK
)

ax1.set_xlim(-1.65, 1.60)
ax1.set_ylim(-1.35, 1.35)
ax1.set_aspect("equal")

# Informal economy bar
ax2 = fig.add_subplot(gs[1, 1:3])
x2 = np.arange(len(informal_df))
bars2 = ax2.bar(
    x2,
    informal_df["percent"],
    color=informal_colors,
    edgecolor="white",
    width=0.60
)

ax2.set_title(
    "Informal economy and formal-visibility constraints",
    fontsize=13,
    fontweight="bold",
    color=DARK
)
ax2.set_ylabel("%")
ax2.set_ylim(0, 92)
ax2.set_xticks(x2)
ax2.set_xticklabels(informal_df["indicator"], fontsize=8.5)

add_bar_labels_vertical(ax2, bars2, informal_df["label"].tolist(), offset=1.4, fontsize=9)

clean_axis(ax2)
ax2.grid(axis="x", visible=False)

# Cash behaviour bar
ax3 = fig.add_subplot(gs[1, 3])
x3 = np.arange(len(cash_behaviour_df))
bars3 = ax3.bar(
    x3,
    cash_behaviour_df["percent"],
    color=consumer_colors,
    edgecolor="white",
    width=0.60
)

ax3.set_title(
    "Consumer cash behaviour",
    fontsize=13,
    fontweight="bold",
    color=DARK
)
ax3.set_ylabel("%")
ax3.set_ylim(0, 108)
ax3.set_xticks(x3)
ax3.set_xticklabels(["Cash\nfood", "Cash\nmost", "Prefer\ncash", "Cash +\ndigital"], fontsize=8)

add_bar_labels_vertical(ax3, bars3, cash_behaviour_df["label"].tolist(), offset=1.4, fontsize=9)

clean_axis(ax3)
ax3.grid(axis="x", visible=False)

# Socioeconomic pressure bar
ax4 = fig.add_subplot(gs[2, 1:3])
x4 = np.arange(len(combined_df))
bars4 = ax4.bar(
    x4,
    combined_df["value"],
    color=[BLUE, CYAN, ORANGE, RED],
    edgecolor="white",
    width=0.60
)

ax4.set_title(
    "Socioeconomic pressure indicators",
    fontsize=13,
    fontweight="bold",
    color=DARK
)
ax4.set_ylabel("%")
ax4.set_ylim(0, 55)
ax4.set_xticks(x4)
ax4.set_xticklabels(combined_df["metric"], fontsize=8.5)

add_bar_labels_vertical(
    ax4,
    bars4,
    [f"{v:.1f}%" for v in combined_df["value"]],
    offset=1.0,
    fontsize=9
)

clean_axis(ax4)
ax4.grid(axis="x", visible=False)

# AML context factors horizontal bar
ax5 = fig.add_subplot(gs[2, 3])
bars5 = ax5.barh(
    context_df["factor"],
    context_df["percent"],
    color=[BLUE, CYAN, GREEN, ORANGE, RED, PURPLE],
    edgecolor="white",
    height=0.55
)

ax5.set_title(
    "AML false-positive context",
    fontsize=13,
    fontweight="bold",
    color=DARK
)
ax5.set_xlabel("%")
ax5.set_xlim(0, max(context_df["percent"]) + 18)

add_bar_labels_horizontal(
    ax5,
    bars5,
    context_df["label"].tolist(),
    offset=1.0,
    fontsize=8.5
)

ax5.spines["top"].set_visible(False)
ax5.spines["right"].set_visible(False)
ax5.spines["left"].set_visible(False)
ax5.spines["bottom"].set_color("#CBD5E1")
ax5.tick_params(axis="y", length=0, colors="#334155", labelsize=7.5)
ax5.tick_params(axis="x", colors="#334155")
ax5.grid(axis="x", alpha=0.22, linestyle="-", linewidth=0.7)
ax5.set_axisbelow(True)

fig.suptitle(
    "South African AML context: cash intensity, informal economy and socioeconomic behaviour",
    fontsize=18,
    fontweight="bold",
    color=DARK,
    y=0.985
)

fig.text(
    0.5,
    0.945,
    "These indicators motivate context-aware simulation: legitimate cash-based and informal activity can resemble AML risk signals.",
    ha="center",
    fontsize=10.5,
    color=GREY
)

fig.text(
    0.01,
    0.01,
    "Sources: PayInc / BusinessTech; Standard Bank Township Informal Economy Research via BusinessTech; SARB Retail CBDC Background Note 2025; SBV Consumer Cash Survey 2024; Stats SA GHS 2024; Stats SA QLFS Q4 2025.",
    ha="left",
    fontsize=8.3,
    color=GREY
)

plt.savefig(OUT_DIR / "sa_aml_context_dashboard.png", bbox_inches="tight")
plt.show()

# ============================================================
# Export evidence table
# ============================================================
evidence_table = pd.DataFrame({
    "Evidence Area": [
        "Cash intensity",
        "Cash outside commercial banks",
        "Informal economy",
        "Informal economy",
        "Informal economy",
        "Informal economy",
        "Consumer cash behaviour",
        "Consumer cash behaviour",
        "Consumer cash behaviour",
        "Cash and digital coexistence",
        "Social grants",
        "Social grants",
        "Labour market",
        "Labour market"
    ],
    "Statistic": [
        "Total cash and coins in circulation",
        "Individuals and non-bank financial players hold cash",
        "Estimated annual informal economy value",
        "Informal economy share of GNP",
        "Informal economy share of employment",
        "Informal businesses unregistered",
        "Adults using cash for food and groceries",
        "Adults using cash most often for food and groceries",
        "Consumers preferring cash in future",
        "Consumers using both cash and digital",
        "Individuals receiving social grants",
        "Households receiving social grants",
        "Official unemployment rate",
        "Expanded / combined unemployment rate"
    ],
    "Value": [
        "R180bn",
        "R118bn, equal to 65.6% of total cash in circulation",
        "R900bn",
        "12.3%",
        "19.5%",
        "~80%",
        "99%",
        "71%",
        "64%",
        "96%",
        "28.2%",
        "39.1%",
        "31.4%",
        "42.1%"
    ],
    "Source": [
        "PayInc, reported by BusinessTech / The Outlier",
        "PayInc, reported by BusinessTech / The Outlier",
        "BusinessTech reporting on Standard Bank Township Informal Economy Research",
        "BusinessTech reporting on Standard Bank Township Informal Economy Research",
        "BusinessTech reporting on Standard Bank Township Informal Economy Research",
        "BusinessTech reporting on Standard Bank Township Informal Economy Research",
        "SARB Retail CBDC Background Note 2025",
        "SARB Retail CBDC Background Note 2025",
        "SBV Consumer Cash Survey White Paper 2024",
        "SBV Consumer Cash Survey White Paper 2024",
        "Stats SA General Household Survey 2024",
        "Stats SA General Household Survey 2024",
        "Stats SA QLFS Q4 2025",
        "Stats SA QLFS Q4 2025"
    ],
    "AML Interpretation": [
        "Confirms that South Africa remains cash intensive.",
        "Large share of cash outside banks creates limited visibility and informal circulation.",
        "Supports the relevance of informal economy variables in simulation.",
        "Shows structural importance of informal activity.",
        "Shows informal activity is employment-relevant and not marginal.",
        "Unregistered businesses weaken standard formal profiling assumptions.",
        "Everyday cash use remains widespread even in basic retail categories.",
        "Cash remains dominant in essential spending categories.",
        "Cash preference supports persistence of cash-heavy transaction behaviour.",
        "Models should allow mixed cash-digital behaviour rather than binary assumptions.",
        "Grant-related inflows can create regular low-value and cash-out transaction patterns.",
        "Grant reliance affects household transaction behaviour and account usage.",
        "High unemployment contributes to irregular income and unstable transaction behaviour.",
        "Broader labour-market stress increases informal and cash-based economic participation."
    ]
})

print(evidence_table)
evidence_table.to_csv(OUT_DIR / "sa_aml_context_evidence_table.csv", index=False)

print(f"\nSaved plots and evidence table to: {OUT_DIR.resolve()}")